# ADVC W1d: why is forward+backward 9-14x slower at batch 32 than at 16? + loader rate (Kaggle T4)

After W1c (RESULTS_LOG R-003): fwd+bwd passes/s dropped from ~134 (bs 16) to ~14 (bs 32) at every precision, fp32 included. Hypothesis: `cudnn.deterministic=True` (new harness only) picks a slow patch-embedding Conv2d backward.

**Before running:** GPU T4 x2, Internet On, Add Input **jainmanav20/advc-results** and **albert1994/tinyimagenet**. Then **Save Version -> Save & Run All (Commit)**.

| Part | What | Estimate (unmeasured) |
|---|---|---|
| A | fwd+bwd passes/s, fp32 and int4, batch 16/24/32/48/64 x 3 cuDNN modes | ~2-4 min GPU |
| B | patch-embedding Conv2d alone, backward-to-input, same modes | <1 min GPU |
| C | are input gradients bit-reproducible at batch 32 in each mode? | <1 min GPU |
| D | steady-state loader rate, val (eval transform) and train (train transform), 2 and 4 workers | ~2-3 min CPU |

Total ~3-5 min GPU, ~10-14 min wall (incl. ~2 min dataset prep). Copy the last cell's output back to Claude.

In [ ]:
# 1. Setup
import os, subprocess, sys, time, glob, json
BRANCH = 'revision-w1'
W = '/kaggle/working/w1d'
os.makedirs(W, exist_ok=True)
if not os.path.isdir('/kaggle/working/ADVC'):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, 'https://github.com/Jmanav/ADVC.git', '/kaggle/working/ADVC'], check=True)
os.chdir('/kaggle/working/ADVC')
GIT = subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout.strip()
print('git:', GIT)
!pip install -q --no-deps bitsandbytes==0.50.2
!pip install -q torchattacks
import torch, bitsandbytes
assert bitsandbytes.__version__ == '0.50.2'
hits = glob.glob('/kaggle/input/**/at_int8_epoch01.pt', recursive=True)
assert hits, 'No checkpoints found: attach jainmanav20/advc-results'
CKPT_ROOT = os.path.dirname(os.path.dirname(hits[0]))
os.makedirs('results/checkpoints', exist_ok=True)
if not os.path.exists('results/checkpoints/tiny-imagenet'):
    os.symlink(CKPT_ROOT, 'results/checkpoints/tiny-imagenet')
SRC = '/kaggle/input/datasets/albert1994/tinyimagenet/tiny-imagenet-200'
if not os.path.isdir(SRC):
    cands = [p for p in glob.glob('/kaggle/input/**/tiny-imagenet-200', recursive=True)
             if os.path.isfile(os.path.join(p, 'val', 'val_annotations.txt'))]
    assert cands, 'TinyImageNet not found: attach albert1994/tinyimagenet'
    SRC = cands[0]
if not os.path.isdir('/kaggle/working/tiny_if/val_if'):
    r = subprocess.run([sys.executable, 'scripts/prepare_tiny_imagenet.py', '--root', SRC, '--out', '/kaggle/working/tiny_if'], capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:])
assert os.path.isdir('/kaggle/working/tiny_if/train_if')

TIMES = {}
def sh(name, cmd):
    t0 = time.time()
    with open(f'{W}/{name}.log', 'w') as log:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            log.write(line); print(line, end='')
        p.wait()
    TIMES[name] = {'seconds': round(time.time() - t0, 1), 'exit': p.returncode}
    print('[' + name + ']', TIMES[name])

In [ ]:
%%writefile /kaggle/working/w1d/p5_gpu.py
# Parts A-C. Each timing: 2 warm-up + 5 timed steps of forward + backward-to-input (the attack step).
import json, sys, time
sys.path.insert(0, '.')
import torch
import torch.nn.functional as F
from models.loader import load_config, load_model

cfg = load_config('configs/base.yaml')
MODES = [('det1_bench0', True, False), ('det0_bench0', False, False), ('det0_bench1', False, True)]
BS = [16, 24, 32, 48, 64]

def set_mode(det, bench):
    torch.backends.cudnn.deterministic = det
    torch.backends.cudnn.benchmark = bench
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

def timed(step, n_img, warm=2, reps=5):
    for _ in range(warm):
        step()
    torch.cuda.synchronize(); t = time.time()
    for _ in range(reps):
        step()
    torch.cuda.synchronize()
    return round(reps * n_img / (time.time() - t), 1)

out = {'A_fwd_bwd_passes_per_s': {}, 'B_conv_bwd_images_per_s': {}, 'C_repeat_grad_bs32': {}}
torch.manual_seed(0)
for prec in ['fp32', 'int4']:
    m = load_model('deit_small', prec, cfg, device='cuda').eval()
    for name, det, bench in MODES:
        set_mode(det, bench)
        for bs in BS:
            x = torch.randn(bs, 3, 224, 224, device='cuda')
            y = torch.randint(0, 200, (bs,), device='cuda')
            def step():
                xg = x.clone().requires_grad_(True)
                torch.autograd.grad(F.cross_entropy(m(xg).float(), y), xg)
            r = timed(step, bs)
            out['A_fwd_bwd_passes_per_s'][prec + '|' + name + '|bs' + str(bs)] = r
            print('A', prec, name, bs, r, flush=True)
    if prec == 'fp32':
        for name, det, bench in MODES:
            set_mode(det, bench)
            torch.manual_seed(1)
            x = torch.randn(32, 3, 224, 224, device='cuda')
            y = torch.randint(0, 200, (32,), device='cuda')
            gs = []
            for _ in range(2):
                xg = x.clone().requires_grad_(True)
                g, = torch.autograd.grad(F.cross_entropy(m(xg).float(), y), xg)
                gs.append(g)
            out['C_repeat_grad_bs32'][name] = {'bit_equal': bool(torch.equal(gs[0], gs[1])),
                                               'max_abs_diff': float((gs[0] - gs[1]).abs().max()),
                                               'sign_agree': float((torch.sign(gs[0]) == torch.sign(gs[1])).float().mean())}
            print('C', name, out['C_repeat_grad_bs32'][name], flush=True)
    del m
    torch.cuda.empty_cache()

conv = torch.nn.Conv2d(3, 384, kernel_size=16, stride=16).cuda()
for name, det, bench in MODES:
    set_mode(det, bench)
    for bs in [16, 32, 64]:
        x = torch.randn(bs, 3, 224, 224, device='cuda')
        def step():
            xg = x.clone().requires_grad_(True)
            torch.autograd.grad(conv(xg).sum(), xg)
        r = timed(step, bs)
        out['B_conv_bwd_images_per_s'][name + '|bs' + str(bs)] = r
        print('B', name, bs, r, flush=True)
print('P5JSON ' + json.dumps(out))
json.dump(out, open('/kaggle/working/w1d/p5_gpu.json', 'w'), indent=1)

In [ ]:
# 2. Run parts A-C (~3-5 min GPU)
sh('p5_gpu', [sys.executable, f'{W}/p5_gpu.py'])

In [ ]:
%%writefile /kaggle/working/w1d/p6_loader.py
# Part D (CPU): steady-state DataLoader rate. First batch (worker start-up) excluded from the timing.
import json, os, sys, time
sys.path.insert(0, '.')
import torch
from torch.utils.data import DataLoader, Subset
from models.loader import load_config
from utils.datasets import build_eval_transform, build_train_transform, build_remapped_folder

cfg = load_config('configs/base.yaml')
out = {'cpu_count': os.cpu_count()}
for split, root, tf, bs in [('val_eval', cfg['dataset']['val_dir'], build_eval_transform(cfg), 32),
                            ('train_train', cfg['dataset']['train_dir'], build_train_transform(cfg), 16)]:
    ds = build_remapped_folder(root, tf, cfg)
    g = torch.Generator().manual_seed(0)
    idx = torch.randperm(len(ds), generator=g)[:1 + 40 * bs].tolist()
    for nw in [2, 4]:
        dl = DataLoader(Subset(ds, idx), batch_size=bs, shuffle=False, num_workers=nw)
        it = iter(dl)
        next(it)
        t = time.time(); n = 0
        for xb, yb in it:
            n += xb.shape[0]
        dt = time.time() - t
        key = split + '|workers' + str(nw)
        out[key] = {'images': n, 'seconds': round(dt, 2), 'images_per_s': round(n / dt, 1)}
        print(key, out[key], flush=True)
print('P6JSON ' + json.dumps(out))
json.dump(out, open('/kaggle/working/w1d/p6_loader.json', 'w'), indent=1)

In [ ]:
# 3. Run part D (CPU, ~2-3 min)
sh('p6_loader', [sys.executable, f'{W}/p6_loader.py'])

In [ ]:
# 4. SUMMARY: copy everything this cell prints back to Claude (also /kaggle/working/w1d/summary.txt). Not truncated.
L = ['git: ' + GIT, 'torch ' + torch.__version__ + ' | bnb ' + bitsandbytes.__version__ + ' | ' + torch.cuda.get_device_name(0)]
L.append('== wall times / exit codes ==')
L.append(json.dumps(TIMES))
for name in ['p5_gpu', 'p6_loader']:
    p = f'{W}/{name}.json'
    L.append('== ' + name + ' ==')
    L.append(open(p).read() if os.path.exists(p) else 'no JSON; log tail:\n' + ''.join(open(f'{W}/{name}.log').readlines()[-40:]))
text = '\n'.join(L)
open(f'{W}/summary.txt', 'w').write(text)
print(text)